# EV Ecosystem Stock Network Analysis

Source notebook found in the local auto-network project folder.

This public portfolio copy keeps the full notebook source visible on GitHub while removing execution outputs, execution counts, and environment-specific metadata.

# EV Ecosystem Stock Network Analysis

This project studies the electric vehicle (EV) ecosystem as a financial network. The objective is not to forecast stock prices, but to understand how companies in the sector move together after removing broad market exposure.

Each company is represented as a node. Edges are created when two companies have a strong correlation in their market-neutral returns. The network is then used to identify central firms, hubs, communities, and possible channels of sector risk transmission.

The analysis is useful for portfolio and risk analysis because highly connected stocks may offer less diversification, while central or bridge firms can indicate areas where shocks may spread through the sector.


In [ ]:
# If running this notebook in a new environment, install the required packages first.
# !pip install yfinance networkx numpy pandas matplotlib scikit-learn node2vec python-louvain


In [ ]:
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import yfinance as yf
from node2vec import Node2Vec
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    normalized_mutual_info_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import community as community_louvain

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)


## 1. Data and Market Universe

The sample is restricted to companies that belong to the EV ecosystem rather than the entire stock market. This gives the network an economic interpretation: firms are linked through manufacturing, suppliers, semiconductors, batteries, materials, software, and mobility platforms.

The market benchmark is `SPY`. It is included so that broad market movement can be removed before the network is constructed.


In [ ]:
start_date = "2022-01-01"
end_date = "2026-07-11"  # yfinance treats this as an exclusive end date.
market = "SPY"

tickers = [
    # Automakers
    "TSLA", "TM", "VOW.DE", "BMW.DE", "MBGYY", "F", "GM", "STLA", "HMC",
    "005380.KS", "BYDDY", "0175.HK", "NIO", "LI", "XPEV", "RIVN", "LCID",

    # Tier-1 suppliers
    "MGA", "APTV", "BWA", "LEA", "DNZOY", "VLEEY",

    # Semiconductors
    "NVDA", "QCOM", "AMD", "INTC", "TXN", "NXPI", "STM", "IFNNY", "TSM",

    # Batteries and materials
    "ALB", "SQM", "373220.KS", "006400.KS", "6752.T",

    # Mobility and autonomy
    "UBER", "DIDIY", "MBLY", "BIDU", "GOOGL",
]

data = yf.download(
    tickers + [market],
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False,
)["Close"].sort_index()

print("Download shape:", data.shape)
print("Date range:", data.index.min(), "to", data.index.max())
data.head()


In [ ]:
print("Number of columns:", data.shape[1])
print(data.columns.tolist())

data.head()


## 2. Data Coverage and Cleaning

Before building the network, I check whether each ticker has enough price history. Different listing dates and exchange calendars can create missing values, and sparse data would make correlation estimates unreliable.

Tickers with more than 10 percent missing prices are removed. After that, remaining rows with missing values are dropped so every stock is measured over the same trading dates.


In [ ]:
coverage = pd.DataFrame({
    "nan_share": data.isna().mean(),
    "first_valid": data.apply(lambda s: s.first_valid_index()),
    "last_valid": data.apply(lambda s: s.last_valid_index()),
    "non_null": data.notna().sum(),
}).sort_values("nan_share", ascending=False)

coverage.head(15)


In [ ]:
max_nan_share = 0.10

nan_share = data.isna().mean().sort_values(ascending=False)
keep_cols = nan_share[nan_share <= max_nan_share].index.tolist()
removed_cols = nan_share[nan_share > max_nan_share].index.tolist()

data_filt = data[keep_cols]
data_clean = data_filt.dropna()

print("Tickers kept:", len(keep_cols))
print("Tickers removed:", removed_cols)
print("Original shape:", data.shape)
print("After ticker filtering:", data_filt.shape)
print("After dropping remaining NaNs:", data_clean.shape)
print("Any NaNs left?", data_clean.isna().any().any())
print("Original date range:", data.index.min(), "to", data.index.max())
print("Cleaned date range:", data_clean.index.min(), "to", data_clean.index.max())

nan_share[nan_share > max_nan_share].to_frame("nan_share")


## 3. Log Returns

The network is based on returns, not price levels. I use log returns because they are standard in financial analysis and behave well when returns are aggregated over time.


In [ ]:
returns = np.log(data_clean / data_clean.shift(1)).iloc[1:]

print("Returns shape:", returns.shape)
returns.head()


## 4. Market Neutralization

Raw stock returns often move together because the whole market moves. If the network were built from raw correlations, it would mostly capture market beta rather than EV-sector structure.

To reduce this effect, each stock return is regressed on `SPY`:

$$
r_i = \alpha_i + \beta_i r_{SPY} + \varepsilon_i
$$

The residual $\varepsilon_i$ is the part of the stock return not explained by the broad market benchmark. The network is built from correlations between these residuals.


In [ ]:
market_ret = returns[market]
asset_rets = returns.drop(columns=[market])

X = np.column_stack([np.ones(len(market_ret)), market_ret.to_numpy()])

betas = {}
residuals = pd.DataFrame(index=asset_rets.index)

for col in asset_rets.columns:
    y = asset_rets[col].to_numpy()
    alpha, beta = np.linalg.lstsq(X, y, rcond=None)[0]
    betas[col] = {"alpha": alpha, "beta": beta}
    residuals[col] = y - X @ np.array([alpha, beta])

betas_df = pd.DataFrame(betas).T

print("Residuals shape:", residuals.shape)
betas_df.head()


## 5. Network Construction

The graph is constructed from correlations between market-neutral residual returns.

- Node: one company stock
- Edge: residual correlation with absolute value at least 0.20
- Weight: absolute residual correlation, so graph algorithms receive positive relationship strengths
- `corr` edge attribute: the signed correlation, kept for interpretation
- Direction: undirected, because correlation is symmetric

The analysis focuses on the largest connected component because path-based network measures are easier to interpret inside one connected graph.


In [ ]:
corr = residuals.corr()
threshold = 0.20

corr_thresholded = corr.where(np.abs(corr) >= threshold, other=0.0).copy()
np.fill_diagonal(corr_thresholded.values, 0.0)

A = corr_thresholded.abs()
G = nx.from_pandas_adjacency(A)

for u, v, data_edge in G.edges(data=True):
    signed_corr = float(corr.loc[u, v])
    data_edge["corr"] = signed_corr
    data_edge["weight"] = abs(signed_corr)

largest_cc = max(nx.connected_components(G), key=len)
G_cc = G.subgraph(largest_cc).copy()

negative_edges = sum(1 for _, _, d in G.edges(data=True) if d["corr"] < 0)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Negative-correlation edges kept:", negative_edges)
print("Isolated nodes:", sum(1 for n in G.nodes if G.degree(n) == 0))
print("Connected components:", nx.number_connected_components(G))
print("Largest CC nodes:", G_cc.number_of_nodes())
print("Largest CC edges:", G_cc.number_of_edges())


In [ ]:
threshold_grid = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

rows = []
for th in threshold_grid:
    A_tmp = corr.where(np.abs(corr) >= th, other=0.0).abs()
    np.fill_diagonal(A_tmp.values, 0.0)
    G_tmp = nx.from_pandas_adjacency(A_tmp)

    largest_component = max(nx.connected_components(G_tmp), key=len)
    rows.append({
        "threshold": th,
        "nodes": G_tmp.number_of_nodes(),
        "edges": G_tmp.number_of_edges(),
        "isolates": sum(1 for n in G_tmp.nodes if G_tmp.degree(n) == 0),
        "components": nx.number_connected_components(G_tmp),
        "largest_cc_nodes": len(largest_component),
        "largest_cc_share": len(largest_component) / G_tmp.number_of_nodes(),
        "density": nx.density(G_tmp),
    })

threshold_check = pd.DataFrame(rows)
threshold_check


The threshold check is used to choose a cutoff that removes weak correlations without breaking the graph into too many small pieces. A threshold of 0.20 keeps a large connected component while filtering out weaker relationships.


In [ ]:
nx.write_gexf(G_cc, "G_cc.gexf")
nx.write_gexf(G, "G_full.gexf")

print("Saved: G_cc.gexf and G_full.gexf")


## Gephi Visualization

The graph is also shown using the Gephi layout exported from the network. This visualization is useful for presentation because it makes the central core, bridge positions, and community-like structure easier to see than a basic notebook layout.


In [ ]:
from pathlib import Path
import matplotlib.image as mpimg

gephi_image_path = Path("web .png")

if gephi_image_path.exists():
    img = mpimg.imread(gephi_image_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Gephi network layout")
    plt.show()
else:
    print(f"Gephi image not found: {gephi_image_path}")


In [ ]:
pos = nx.spring_layout(G_cc, seed=42, weight="weight")
node_sizes = [120 + 30 * G_cc.degree(n) for n in G_cc.nodes()]
edge_widths = [0.5 + 2.5 * G_cc[u][v]["weight"] for u, v in G_cc.edges()]

plt.figure(figsize=(10, 7))
nx.draw_networkx_edges(G_cc, pos, alpha=0.35, width=edge_widths)
nx.draw_networkx_nodes(G_cc, pos, node_size=node_sizes, node_color="#4C78A8", alpha=0.90)
nx.draw_networkx_labels(G_cc, pos, font_size=8)
plt.title("EV ecosystem residual-correlation network")
plt.axis("off")
plt.show()


## 6. Degree Distribution and Hubs

Degree counts how many strong relationships each stock has. A hub is a stock with unusually many connections compared with the rest of the network.


In [ ]:
degree_sample = np.array([d for _, d in G_cc.degree()])

print("G_cc nodes:", G_cc.number_of_nodes())
print("G_cc edges:", G_cc.number_of_edges())
print("Degree min/mean/max:", degree_sample.min(), degree_sample.mean(), degree_sample.max())

x = np.sort(np.unique(degree_sample))
ccdf = np.array([(degree_sample >= k).mean() for k in x])

plt.figure(figsize=(7, 5))
plt.loglog(x, ccdf, marker="o", linestyle="none")
plt.xlabel("Degree k (log)")
plt.ylabel("CCDF P(K >= k) (log)")
plt.title("Degree distribution")
plt.grid(True, which="both")
plt.show()


The degree distribution is right-skewed: most firms have a moderate number of strong links, while a smaller group of firms is much more connected. In a financial-risk interpretation, these highly connected firms are important because their movements are shared with many other stocks in the sector.


In [ ]:
hub_percentile = 85
hub_threshold = np.percentile(degree_sample, hub_percentile)
hubs = [n for n, d in G_cc.degree() if d >= hub_threshold]
hub_subgraph = G_cc.subgraph(hubs).copy()

print(f"Hub degree threshold ({hub_percentile}th percentile):", hub_threshold)
print("Hubs:", hubs)
print("Number of hubs:", len(hubs))
print("Hub subgraph nodes:", hub_subgraph.number_of_nodes())
print("Hub subgraph edges:", hub_subgraph.number_of_edges())
print("Hub subgraph connected components:", nx.number_connected_components(hub_subgraph))

if hub_subgraph.number_of_nodes() > 0:
    hub_comp_sizes = sorted([len(c) for c in nx.connected_components(hub_subgraph)], reverse=True)
    print("Hub component sizes:", hub_comp_sizes)


Hubs are defined as the top 15 percent of stocks by degree. If these hubs are connected to each other, they form a dense core rather than acting as separate influential firms.


## 7. Centrality Analysis

Centrality measures describe different types of importance.

- Degree: many direct connections
- Strength: sum of weighted connections
- Eigenvector centrality: connections to other important nodes
- Betweenness: bridge position between parts of the graph
- PageRank: global importance based on connections to important nodes

Betweenness is calculated without weights because correlation strength is a similarity measure, not a distance.


In [ ]:
metrics_df = pd.DataFrame(index=G_cc.nodes())
metrics_df["degree"] = pd.Series(dict(G_cc.degree()))
metrics_df["strength"] = pd.Series(dict(G_cc.degree(weight="weight")))
metrics_df["degree_centrality"] = pd.Series(nx.degree_centrality(G_cc))
metrics_df["betweenness"] = pd.Series(nx.betweenness_centrality(G_cc, weight=None))
metrics_df["eigenvector"] = pd.Series(nx.eigenvector_centrality(G_cc, max_iter=5000, weight="weight"))
metrics_df["pagerank"] = pd.Series(nx.pagerank(G_cc, weight="weight"))

metrics_df.sort_values("eigenvector", ascending=False).head(15)


In [ ]:
def top_n_ranks(centrality_dict, n=10):
    return sorted(centrality_dict, key=centrality_dict.get, reverse=True)[:n]

top_degree = top_n_ranks(nx.degree_centrality(G_cc), n=10)
top_eigenvector = top_n_ranks(nx.eigenvector_centrality(G_cc, max_iter=5000, weight="weight"), n=10)
top_betweenness = top_n_ranks(nx.betweenness_centrality(G_cc, weight=None), n=10)
top_pagerank = top_n_ranks(nx.pagerank(G_cc, weight="weight"), n=10)

rank_table = pd.DataFrame({
    "degree": top_degree,
    "eigenvector": top_eigenvector,
    "betweenness": top_betweenness,
    "pagerank": top_pagerank,
})

super_central = sorted(set(top_degree) & set(top_eigenvector) & set(top_betweenness) & set(top_pagerank))

print("Common to all four top-10 lists:", super_central)
rank_table


In [ ]:
centrality_cols = ["degree_centrality", "betweenness", "eigenvector", "pagerank"]
corr_c = metrics_df[centrality_cols].corr(method="kendall")

plt.figure(figsize=(6, 5))
plt.imshow(corr_c.values)
plt.xticks(range(len(centrality_cols)), centrality_cols, rotation=45, ha="right")
plt.yticks(range(len(centrality_cols)), centrality_cols)

for i in range(len(centrality_cols)):
    for j in range(len(centrality_cols)):
        plt.text(j, i, f"{corr_c.values[i, j]:.2f}", ha="center", va="center")

plt.title("Kendall correlation between centrality rankings")
plt.tight_layout()
plt.show()

corr_c


The centrality measures are positively related, but they do not measure exactly the same role. Degree and eigenvector centrality focus on connectedness and core position, while betweenness highlights firms that act as bridges between groups.


## 8. Clustering, Triangles, and Random Graph Comparison

A triangle means that three stocks are all mutually connected. A high triangle count and a high clustering coefficient indicate that firms form tight groups rather than isolated pairwise relationships.


In [ ]:
triangles = nx.triangles(G_cc)
tri_sample = np.array(list(triangles.values()))

clustering = nx.clustering(G_cc)
clustering_values = np.array(list(clustering.values()))

structure_summary = pd.DataFrame({
    "metric": [
        "triangles_min",
        "triangles_mean",
        "triangles_max",
        "average_clustering",
        "density",
    ],
    "value": [
        tri_sample.min(),
        tri_sample.mean(),
        tri_sample.max(),
        clustering_values.mean(),
        nx.density(G_cc),
    ],
})

structure_summary


In [ ]:
x_cc = np.sort(clustering_values)
y_cc = np.arange(1, len(clustering_values) + 1) / len(clustering_values)

deg_dict = dict(G_cc.degree())
deg_vals = np.array([deg_dict[n] for n in G_cc.nodes()])
cc_vals = np.array([clustering[n] for n in G_cc.nodes()])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(x_cc, y_cc, marker="o", linestyle="none")
axes[0].set_xlabel("Clustering coefficient")
axes[0].set_ylabel("ECDF")
axes[0].set_title("Local clustering coefficient")
axes[0].grid(True)

axes[1].scatter(deg_vals, cc_vals, alpha=0.6)
axes[1].set_xlabel("Degree k")
axes[1].set_ylabel("Clustering coefficient")
axes[1].set_title("Clustering coefficient vs degree")
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
def ecdf_positive(values):
    values = np.asarray(list(values))
    values = np.sort(values[values > 0])
    if len(values) == 0:
        return np.array([]), np.array([])
    x = np.unique(values)
    y = np.searchsorted(values, x, side="right") / len(values)
    return x, y

n_nodes = G_cc.number_of_nodes()
er_prob = nx.density(G_cc)
rng = np.random.default_rng(42)

x_obs, y_obs = ecdf_positive(nx.triangles(G_cc).values())

plt.figure(figsize=(7, 5))
for seed in rng.integers(0, 1_000_000_000, size=5):
    G_er = nx.fast_gnp_random_graph(n_nodes, er_prob, seed=int(seed))
    x_er, y_er = ecdf_positive(nx.triangles(G_er).values())
    if len(x_er):
        plt.loglog(x_er, y_er, alpha=0.35, color="gray")

if len(x_obs):
    plt.loglog(x_obs, y_obs, "-o", linewidth=3, label="Observed")

plt.xlabel("Triangles per node (log)")
plt.ylabel("ECDF (log)")
plt.title("Triangles per node: observed vs Erdos-Renyi")
plt.grid(True, which="both")
plt.legend()
plt.show()


The random graph comparison gives a baseline. Erdos-Renyi graphs with the same number of nodes and similar density do not reproduce the same clustering pattern as the observed EV network. This supports the interpretation that the observed structure is not just a random set of correlations.


## 9. Community Detection

Louvain community detection groups stocks that are more strongly connected to each other than to the rest of the network. In this project, communities can be interpreted as market-implied groups within the EV ecosystem.


In [ ]:
partition = community_louvain.best_partition(G_cc, weight="weight", random_state=42)
communities = pd.Series(partition, name="community")

metrics_df = metrics_df.join(communities)

print("Community counts:")
print(communities.value_counts().sort_index())

community_members = (
    metrics_df[["community", "degree", "eigenvector", "betweenness", "pagerank"]]
    .sort_values(["community", "degree"], ascending=[True, False])
)
community_members.head(25)


### What the Communities Represent

Based on the current run, Louvain finds four communities. The community numbers are just labels; the meaning comes from the companies inside each group.

**Community 0: Korean auto/battery cluster**

`005380.KS`, `006400.KS`, and `373220.KS` form a small Korea-linked group. This mainly reflects Hyundai and Korean battery-related firms.

**Community 1: Semiconductor cluster**

`AMD`, `IFNNY`, `INTC`, `NVDA`, `NXPI`, `QCOM`, `STM`, `TSM`, and `TXN` form the semiconductor group. This makes sense because these firms share technology-cycle and chip-supply exposure.

**Community 2: EV/new energy/mobility cluster**

`0175.HK`, `ALB`, `BIDU`, `BYDDY`, `DIDIY`, `LCID`, `LI`, `NIO`, `RIVN`, `SQM`, `TSLA`, and `XPEV` form a group connected to EV manufacturers, Chinese EV names, battery materials, and mobility/autonomy exposure.

**Community 3: Traditional automakers and Tier-1 suppliers**

`6752.T`, `APTV`, `BMW.DE`, `BWA`, `DNZOY`, `F`, `GM`, `HMC`, `LEA`, `MBGYY`, `MGA`, `STLA`, `TM`, `VLEEY`, and `VOW.DE` form the largest community. This group combines traditional automakers with major suppliers, which supports the interpretation that suppliers and manufacturers form the core of the EV ecosystem network.

Overall, the communities show that the EV sector is not one uniform group. The network separates into subgroups based on business role, region, and technology exposure. For portfolio analysis, stocks inside the same community may offer weaker diversification because their market-neutral returns are more closely connected.


In [ ]:
pos_comm = nx.spring_layout(G_cc, seed=42, weight="weight")
node_colors = [partition[n] for n in G_cc.nodes()]

plt.figure(figsize=(10, 7))
nx.draw_networkx_edges(G_cc, pos_comm, alpha=0.35)
nx.draw_networkx_nodes(G_cc, pos_comm, node_size=500, node_color=node_colors, cmap="tab10")
nx.draw_networkx_labels(G_cc, pos_comm, font_size=8)
plt.title("Louvain communities in the EV stock network")
plt.axis("off")
plt.show()


The communities show that stocks are not arranged as one uniform sector. They split into groups that broadly reflect business roles, regions, and technology exposure, such as manufacturers and suppliers, semiconductors, battery/material firms, and EV-focused companies.


## 10. Node2Vec Embeddings

Node2Vec learns a vector representation for each stock by simulating random walks on the graph. Stocks that occupy similar positions in the network should have similar embeddings.

The embeddings are used in two ways:

1. Clustering stocks with KMeans and comparing the result with Louvain communities.
2. Predicting whether a relationship exists between two stocks.


In [ ]:
node2vec = Node2Vec(
    G_cc,
    dimensions=32,
    walk_length=20,
    num_walks=200,
    workers=1,
    seed=42,
    weight_key="weight",
    quiet=True,
)

model = node2vec.fit(window=10, min_count=1, batch_words=64)
emb = {str(n): model.wv[str(n)] for n in G_cc.nodes()}

print("Embedding dimension:", len(next(iter(emb.values()))))


In [ ]:
nodes_list = list(G_cc.nodes())
X_emb = np.vstack([emb[str(n)] for n in nodes_list])

n_communities = len(set(partition.values()))
kmeans = KMeans(n_clusters=n_communities, random_state=42, n_init="auto")
embedding_clusters = kmeans.fit_predict(X_emb)

louvain_labels = np.array([partition[n] for n in nodes_list])
nmi = normalized_mutual_info_score(louvain_labels, embedding_clusters)

print("KMeans clusters:", n_communities)
print("NMI between Louvain and Node2Vec/KMeans:", nmi)

pd.crosstab(
    pd.Series(louvain_labels, name="Louvain"),
    pd.Series(embedding_clusters, name="KMeans"),
)


A high NMI score means that the embedding-based clusters are close to the Louvain communities. This suggests that Node2Vec captures meaningful graph structure rather than only producing arbitrary vectors.


## 11. Link Prediction

The link-prediction task tests whether the network structure can help distinguish real stock relationships from true non-edges.

To avoid data leakage, test edges are removed before Node2Vec is trained. Negative examples are sampled from pairs that are non-edges in the original graph, so removed positive test edges are not accidentally labelled as negatives.


In [ ]:
def hadamard(u, v, embedding):
    return embedding[str(u)] * embedding[str(v)]


def l1_distance(u, v, embedding):
    return np.abs(embedding[str(u)] - embedding[str(v)])


def fit_node2vec_embeddings(G_train, dimensions=32, walk_length=20, num_walks=200, seed=42):
    n2v = Node2Vec(
        G_train,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=1,
        seed=seed,
        weight_key="weight",
        quiet=True,
    )
    model = n2v.fit(window=10, min_count=1, batch_words=64)
    return {str(n): model.wv[str(n)] for n in G_train.nodes()}


def make_edge_features(edges_pos, edges_neg, embedding, operator="hadamard"):
    operators = {"hadamard": hadamard, "l1": l1_distance}
    op = operators[operator]

    X_pos = np.vstack([op(u, v, embedding) for u, v in edges_pos])
    X_neg = np.vstack([op(u, v, embedding) for u, v in edges_neg])
    X = np.vstack([X_pos, X_neg])
    y = np.hstack([np.ones(len(edges_pos)), np.zeros(len(edges_neg))])
    return X, y


def best_f1_threshold(y_true, probabilities, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)

    best_threshold, best_f1 = 0.5, -1.0
    for threshold_value in grid:
        predictions = (probabilities >= threshold_value).astype(int)
        score = f1_score(y_true, predictions)
        if score > best_f1:
            best_threshold, best_f1 = threshold_value, score

    return float(best_threshold), float(best_f1)


def leakage_free_link_prediction(G, operator="hadamard", test_frac=0.30, seed=42):
    positive_edges = list(G.edges())
    pos_train, pos_test = train_test_split(
        positive_edges,
        test_size=test_frac,
        random_state=seed,
    )

    true_non_edges = list(nx.non_edges(G))
    neg_train, neg_test = train_test_split(
        true_non_edges,
        train_size=len(pos_train),
        test_size=len(pos_test),
        random_state=seed,
    )

    G_train = G.copy()
    G_train.remove_edges_from(pos_test)

    emb_train = fit_node2vec_embeddings(
        G_train,
        dimensions=32,
        walk_length=20,
        num_walks=200,
        seed=seed,
    )

    X_train, y_train = make_edge_features(pos_train, neg_train, emb_train, operator=operator)
    X_test, y_test = make_edge_features(pos_test, neg_test, emb_train, operator=operator)

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)),
    ])
    clf.fit(X_train, y_train)

    p_train = clf.predict_proba(X_train)[:, 1]
    p_test = clf.predict_proba(X_test)[:, 1]

    threshold_best, f1_train_best = best_f1_threshold(y_train, p_train)
    yhat_test = (p_test >= threshold_best).astype(int)

    return {
        "operator": operator,
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "pos_train": len(pos_train),
        "pos_test": len(pos_test),
        "threshold_train_F1": threshold_best,
        "F1_train_best": f1_train_best,
        "ROC_AUC": roc_auc_score(y_test, p_test),
        "PR_AUC": average_precision_score(y_test, p_test),
        "F1_test": f1_score(y_test, yhat_test),
    }


In [ ]:
res_hadamard = leakage_free_link_prediction(G_cc, operator="hadamard", test_frac=0.30, seed=42)
res_l1 = leakage_free_link_prediction(G_cc, operator="l1", test_frac=0.30, seed=42)

pd.DataFrame([res_hadamard, res_l1])[
    ["operator", "pos_train", "pos_test", "threshold_train_F1", "ROC_AUC", "PR_AUC", "F1_test"]
]


In [ ]:
seeds = list(range(10))

rows = []
for seed in seeds:
    rows.append(leakage_free_link_prediction(G_cc, operator="l1", test_frac=0.30, seed=seed))
    rows.append(leakage_free_link_prediction(G_cc, operator="hadamard", test_frac=0.30, seed=seed))

link_prediction_results = pd.DataFrame(rows)
link_prediction_summary = link_prediction_results.groupby("operator")[["ROC_AUC", "PR_AUC", "F1_test"]].agg(["mean", "std"])

link_prediction_summary


The multi-seed evaluation checks that the link-prediction result is not driven by one lucky random walk initialization or one particular train-test split. The result should be interpreted as evidence that the network has learnable structure, not as a stock-price forecasting model.


## Key Results from the Capped Run

This table collects the main numerical results from the final run, using data capped at `2026-07-11`.


In [ ]:
key_results = pd.DataFrame({
    "item": [
        "cleaned date range",
        "stocks after cleaning",
        "full graph nodes",
        "full graph edges",
        "largest connected component",
        "average clustering",
        "hub threshold",
        "hubs",
        "super-central firms",
        "Node2Vec/Louvain NMI",
        "best mean ROC-AUC",
        "best mean PR-AUC",
        "best mean F1",
    ],
    "value": [
        f"{data_clean.index.min().date()} to {data_clean.index.max().date()}",
        residuals.shape[1],
        G.number_of_nodes(),
        G.number_of_edges(),
        f"{G_cc.number_of_nodes()} nodes, {G_cc.number_of_edges()} edges",
        float(structure_summary.loc[structure_summary["metric"] == "average_clustering", "value"].iloc[0]),
        f">= {hub_threshold:.0f} degree",
        ", ".join(hubs),
        ", ".join(super_central),
        float(nmi),
        float(link_prediction_summary.loc["l1", ("ROC_AUC", "mean")]),
        float(link_prediction_summary.loc["l1", ("PR_AUC", "mean")]),
        float(link_prediction_summary.loc["l1", ("F1_test", "mean")]),
    ],
})

key_results


## 12. Final Summary

Using data capped at `2026-07-11`, the cleaned sample contains 41 stocks after removing the market benchmark. The largest connected component has 39 nodes and 193 edges, so almost all retained relationships belong to one main EV-sector network.

The network is highly clustered, with an average clustering coefficient of about 0.725. The hub set is formed by firms with degree at least 16, and in this run the hubs are `MGA`, `VLEEY`, `MBGYY`, `LEA`, `APTV`, `BWA`, and `IFNNY`. The firms that appear across all four top-10 centrality lists are `APTV`, `MBGYY`, `MGA`, and `VLEEY`.

The graph machine-learning results support the same interpretation. Node2Vec and KMeans show strong agreement with Louvain communities, with NMI about 0.941. In the multi-seed link-prediction test, the L1 operator performs best on average, with ROC-AUC about 0.924, PR-AUC about 0.931, and F1 about 0.836.

Overall, the project shows that the EV ecosystem has a structured market-implied dependency network. The results are useful for exploratory portfolio and risk analysis because they highlight firms that are highly connected, firms that sit in the network core, and groups of stocks that may not provide strong diversification from each other.

The main limitation is that the network is based on correlations, so it does not prove causality or direct supply-chain dependence. The results also depend on the selected time period, the ticker universe, the benchmark used for market neutralization, and the correlation threshold. For this reason, the project should be viewed as exploratory financial network and risk analysis rather than a trading strategy.
